# ZuCo Thought Embedding: ZTE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victor-iyi/zte/blob/HEAD/notebooks/zte_colab.ipynb)

**Mount → train → run the study suite → explore → back up to Drive.** Train on a powerful Colab Pro GPU, explore the results **inline** (tables, charts, figures), and keep everything **permanently on Drive** so a dropped runtime never loses work — then run inference locally.

- **Platform-adaptable & auto-accelerated.** `--device auto` (the default) picks **CUDA (Colab GPU) → Cloud TPU (torch_xla) → Apple MPS → CPU**. Nothing to configure.
- **Resumable & runtime-loss-proof.** Every long run is `--resume`-safe. Point `OUT_ROOT` at Drive (Sections 6/6b) to persist runs the instant they finish, and call `backup_to_drive()` (Section 4) anytime for a provenance-stamped archive + a browsable mirror.
- **Reproducible.** Each run keeps its exact resolved `config.yaml`; `zte-pack` archives carry a `PROVENANCE.json`/`PROVENANCE.md` (git commit + package versions + per-run metrics) so any result can be reproduced and trusted.
- **Single fixed seed (42)** by default for clean, comparable runs; bump to multiple seeds where you want confidence intervals.
- **No Colab surprises.** Section 2 sets the env vars Colab leaves unset and fixes the working directory / output paths so the CLIs never error on a fresh runtime.

> ZTE requires **Python 3.14** (Colab ships an older Python), so we use [`uv`](https://docs.astral.sh/uv/) to provision it — one cell, no system changes. All ZTE code therefore runs via `!uv run …` (the 3.14 venv); the plain notebook kernel is used only to read result files for the inline exploration in Section 8b.

**Pick a GPU runtime now:** `Runtime → Change runtime type → T4/A100 GPU` (or `TPU`).

## 1 · Set up (uv provisions Python 3.14 + installs ZTE)

In [ ]:
%%bash
pip install -q uv
# Clone the repo only if we're not already inside it.
[ -f pyproject.toml ] || [ -d zte ] || git clone --depth 1 https://github.com/victor-iyi/zte.git --branch main
[ -f pyproject.toml ] || cd zte
# Provision Python 3.14 + install torch (CUDA wheel on a Colab GPU) and all extras. Cached across runs.
uv python install 3.14

uv sync --all-groups

## 2 · Bootstrap the environment

Colab does not set the env vars headless plotting / tokenizers expect, and the CLIs use paths relative to the repo root. This cell sets those vars for every `!uv run …` subprocess and creates the `res/` output directories, so nothing errors later. Idempotent — safe to re-run.

> **HuggingFace token (optional but recommended).** To silence the *“You are sending unauthenticated requests to the HF Hub”* warning and get higher rate limits + faster downloads for the frozen text encoders (E5 / Qwen in Section 5-vi) and `transformers`, add your token as a Colab secret: **left sidebar → 🔑 → add `HF_TOKEN`** (value from <https://huggingface.co/settings/tokens>) and toggle **notebook access** on. This cell reads it via `google.colab.userdata` and exports `HF_TOKEN` to every subprocess. Without it, downloads still work — just unauthenticated.

In [ ]:
import os

# Enter the repo in the notebook kernel so relative paths + every `!uv run`
# subprocess resolve. This persists across cells (a %%bash `cd` cannot); the setup cell
# above installed ZTE via %%bash.
if os.path.isdir('zte') and not os.path.isfile('pyproject.toml'):
    os.chdir('zte')

# Set in the notebook kernel so every `!uv run` subprocess inherits them.
# MPLBACKEND is FORCED: Colab sets it to an inline backend that crashes a headless subprocess.
os.environ['MPLBACKEND'] = 'Agg'
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('MPLCONFIGDIR', os.path.abspath('res/.cache/matplotlib'))

# Authenticate to the HuggingFace Hub so the frozen text encoders (E5 / Qwen, Section 5-vi) and any
# `transformers` weights download with higher rate limits and no throttling. Reads the `HF_TOKEN`
# Colab secret (left sidebar -> 🔑 -> add `HF_TOKEN`, then toggle notebook access) and exports it so
# every `!uv run` subprocess inherits it. No-op off Colab or when the secret is absent — downloads
# then fall back to slower, rate-limited unauthenticated requests.
try:
    from google.colab import userdata  # type: ignore[import-untyped]

    _hf_token = userdata.get('HF_TOKEN')
except Exception as exc:  # not on Colab, or the secret is missing / not granted to this notebook
    _hf_token = None
    print(
        f'HF_TOKEN unavailable ({type(exc).__name__}) — HuggingFace downloads will be unauthenticated.'
    )
if _hf_token:
    os.environ['HF_TOKEN'] = _hf_token
    print('HF_TOKEN loaded from Colab secrets — authenticated HuggingFace Hub downloads enabled.')

# Create res/ dirs + report the resolved root/accelerator via the 3.14 venv.
!uv run python -c "from zte.utils import bootstrap; import json; print(json.dumps(bootstrap(chdir=True, quiet=True), default=str, indent=1))"

## 3 · Confirm the accelerator — and how ZTE adapts to it
`--device auto` (the default in every cell) picks the best backend — **CUDA → Cloud TPU → Apple MPS → CPU** — and ZTE then tunes itself to that hardware **automatically, without touching accuracy**:

- **A100 / H100 (CUDA, Ampere+):** bf16 mixed precision + **TF32** matmuls (a large, free speedup; fp32 master weights keep accuracy). Older CUDA falls back to fp16 + GradScaler.
- **Cloud TPU (v6e etc.):** bf16 (TPUs are bf16-native) + **static-shape padding** so XLA compiles once instead of recompiling per batch — padded positions are masked out, so results are unchanged.
- **Apple Silicon (MPS):** stable fp32 (MPS autocast is still maturing) — fully GPU-accelerated locally, and the flagship now runs end-to-end here (the `pdist` op was replaced with a portable equivalent).
- **CPU:** fp32, single-process loading.

DataLoader workers are auto-picked per backend too. The cell below prints both what was **detected** and exactly what **ZTE will use**. Everything is overridable via `--precision`, `--num-workers`, `--compile`, `--static-shapes` on `zte-run`/`zte-benchmark`.

In [ ]:
%%bash
uv run python - <<'PY'
import json

from zte.device import auto_num_workers, resolve_device
from zte.utils.env import accelerator_info

info = accelerator_info()
spec = resolve_device('auto')  # what ZTE will use with --device auto
plan = {
    'backend': spec.kind,
    'device': spec.name,
    'autocast_dtype': str(spec.autocast_dtype).replace('torch.', '') if spec.autocast_dtype else 'fp32',
    'mixed_precision': spec.use_amp,      # bf16 on Ampere+/TPU, fp16 on older CUDA, off on MPS/CPU
    'pin_memory': spec.supports_pin_memory,
    'dataloader_workers_auto': auto_num_workers(spec, -1),
    'tf32_matmul': spec.kind == 'cuda',   # Ampere+ (A100/H100): free matmul speedup
    'static_shapes': spec.kind == 'xla',  # TPU only: fixed-length padding (accuracy-neutral)
}
print(json.dumps({'detected': info, 'zte_will_use': plan}, indent=2))
PY


### (Optional) Cloud TPU
On a **TPU** runtime, install `torch_xla` so `--device auto` selects it. `torch_xla` must match the installed torch; this is best-effort (GPU is the primary, tested path). Uncomment to try:

In [ ]:
# !uv pip install -q torch_xla   # then re-run the accelerator cell above; it should report a Cloud TPU

## 4 · Get data + set up permanent Drive backup
**A) Synthetic (default, no dataset).** A fabricated ZuCo tree — validates the whole pipeline in minutes. The `--synthetic` cells below use it automatically; skip this cell if that is all you want.

**B) Real ZuCo + permanent backup.** Mount Drive, read the dataset **directly** from `Sharables/ZTE/ZuCo Dataset` (mounting is faster than re-downloading), and set up the backup target. Everything this session produces is saved under a **date-stamped** folder `Sharables/ZTE/{RUN_DATE}/` — so a dropped Colab runtime never loses finished work. The cell defines `backup_to_drive()`, used throughout. Shareable ZTE folder: <https://drive.google.com/drive/folders/13EYW1h6dHD5E4YoEWNsKe6ZBHmMU_kFQ>.

In [ ]:
from google.colab import drive  # type: ignore[import-untyped]

drive.mount('/gdrive')

import datetime
import glob
import json
import os
import pathlib
import shutil
import subprocess

# --- One shareable ZTE folder holds everything (data + every session's outputs) ---
# https://drive.google.com/drive/folders/13EYW1h6dHD5E4YoEWNsKe6ZBHmMU_kFQ
ZTE_DRIVE = '/gdrive/My Drive/Sharables/ZTE'
DATA_DIR = f'{ZTE_DRIVE}/ZuCo Dataset'  # the ZuCo .mat files on your Drive
# To RESUME an interrupted session, set RESUME_DATE to its date (e.g. '2026-07-12'); else None = today.
RESUME_DATE = None
RUN_DATE = (
    RESUME_DATE or datetime.date.today().isoformat()
)  # groups this session's outputs on Drive
DRIVE_DIR = f'{ZTE_DRIVE}/{RUN_DATE}'  # everything this session produces backs up here
for sub in ('', '/experiments', '/archives'):
    os.makedirs(f'{DRIVE_DIR}{sub}', exist_ok=True)
# Expose paths to %%bash cells (which can't read Python vars) as $DATA_DIR / $DRIVE_DIR / ...
os.environ.update(ZTE_DRIVE=ZTE_DRIVE, DATA_DIR=DATA_DIR, DRIVE_DIR=DRIVE_DIR, RUN_DATE=RUN_DATE)

print('data found :', os.path.isdir(DATA_DIR))
print('backups -> :', DRIVE_DIR)
!ls "{DATA_DIR}" | head


def _real_runs(experiments: str = 'res/experiments') -> list[pathlib.Path]:
    """Non-synthetic run dirs — skips --synthetic smoke runs (missing flag = treated as real)."""
    keep = []
    for mf in glob.glob(f'{experiments}/*/manifest.json'):
        try:
            synthetic = json.load(open(mf)).get('synthetic', False)
        except Exception:
            synthetic = False
        if not synthetic:
            keep.append(pathlib.Path(mf).parent)
    return sorted(keep)


def backup_to_drive(note: str | None = None, include_synthetic: bool = False) -> None:
    """Back up REAL (non-synthetic) runs to Drive. Safe to call anytime / repeatedly.

    Smoke / --synthetic runs are skipped by default (pass include_synthetic=True to force); if there
    are no real runs this is a friendly no-op, so nothing pollutes Drive. Produces a restorable,
    provenance-stamped best-only zip under {DRIVE_DIR}/archives/ and a browsable mirror of the real
    runs (reports, figures, 3-D explorers, best.pt) under {DRIVE_DIR}/experiments/. Real benchmarks
    are written straight to Drive by the benchmark cell (Section 7).
    """
    runs = None if include_synthetic else _real_runs()
    if runs is not None and not runs:
        print(
            'No real (non-synthetic) runs to back up — skipping Drive. Pass include_synthetic=True to force.'
        )
        return
    ts = datetime.datetime.now().strftime('%H%M%S')
    arch = f'{DRIVE_DIR}/archives/zte_{RUN_DATE}_{ts}.zip'
    cmd = ['uv', 'run', 'zte-pack', 'zip', '--all', '--best-only', '--out', arch]
    if not include_synthetic:
        cmd.append('--skip-synthetic')
    if note:
        cmd += ['--note', note]
    print('-> provenance zip:', arch)
    subprocess.run(cmd, check=False)
    ignore = shutil.ignore_patterns('cache', 'tb', 'bundle', 'ckpt_epoch*.pt', 'last.pt')
    dst = pathlib.Path(DRIVE_DIR) / 'experiments'
    if runs is None:
        src = pathlib.Path('res/experiments')
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True, ignore=ignore)
    else:
        for r in runs:
            shutil.copytree(r, dst / r.name, dirs_exist_ok=True, ignore=ignore)
    print(f'backed up {"all" if runs is None else len(runs)} run(s) to', DRIVE_DIR)


def snapshot_to_drive(
    note: str | None = None,
    targets: list[str] | None = None,
    move: bool = False,
    include_synthetic: bool = False,
) -> str:
    """Zip the FULL working state to Drive so you can continue LOCALLY without GPU time.

    Captures res/experiments + res/cache + res/benchmark + res/explorer in ONE provenance-stamped
    zip (the dataset cache means a local session never re-prepares data). --synthetic experiment
    runs are excluded by default (include_synthetic=True to keep them). Download the single file,
    then `zte-pack unpack <zip> --dest res` on your machine to keep exploring / training offline.
    """
    ts = datetime.datetime.now().strftime('%H%M%S')
    out = f'{DRIVE_DIR}/archives/zte_snapshot_{RUN_DATE}_{ts}.zip'
    cmd = ['uv', 'run', 'zte-pack', 'snapshot', *(targets or []), '--out', out]
    if not include_synthetic:
        cmd.append('--skip-synthetic')
    if note:
        cmd += ['--note', note]
    if move:
        cmd.append('--move')
    print('-> full snapshot ->', out)
    subprocess.run(cmd, check=False)
    print('snapshot on Drive:', out)
    return out


def remove_from_res(*names: str) -> None:
    """Easily delete run dirs / res/ subpaths locally to free space (does NOT touch Drive).

    remove_from_res('colab_exp6')                  # a run name under res/experiments/
    remove_from_res('res/benchmark', 'res/cache')   # any res/ subpath
    """
    for n in names:
        p = pathlib.Path(n)
        if not p.exists():
            p = pathlib.Path('res/experiments') / n  # bare run name
        if p.exists():
            shutil.rmtree(p)
            print('removed', p)
        else:
            print('not found:', n)


def restore_from_drive(
    run_date: str | None = None, drive_sub: str = 'experiments', local: str = 'res/experiments'
) -> None:
    """Pull a Drive session's runs back to local so you can resume after a runtime reset.

    Copies {ZTE_DRIVE}/<date>/<drive_sub>/* -> <local>/ (checkpoints, config, eval), then re-run the
    training cell with --resume: finished runs skip, interrupted ones continue from their last checkpoint.
    Defaults to this session's RUN_DATE + the flat experiments/ dir; for the LOSO sweep pass
    restore_from_drive(drive_sub='loso', local='res/experiments/loso').
    """
    date = run_date or RUN_DATE
    src = pathlib.Path(f'{ZTE_DRIVE}/{date}/{drive_sub}')
    if not src.is_dir():
        print('nothing to restore at', src)
        return
    dst = pathlib.Path(local)
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(
        f'restored {len(list(src.iterdir()))} item(s) from {src} -> {dst}. Re-run with --resume to continue.'
    )


def mirror_to_drive(local: str, drive_sub: str | None = None) -> None:
    """Copy a local dir to Drive (browsable), minus heavy transient files (cache/tb/bundle/last.pt/epoch ckpts).

    Persists FULL runs (eval, figures, interactive, COMPARE.html) after training locally with DRIVE_BACKUP.
    e.g. mirror_to_drive('res/experiments/loso', 'loso').
    """
    src = pathlib.Path(local)
    if not src.is_dir():
        print('nothing to mirror at', local)
        return
    dst = pathlib.Path(DRIVE_DIR) / (drive_sub or src.name)
    ignore = shutil.ignore_patterns('cache', 'tb', 'bundle', 'ckpt_epoch*.pt', 'last.pt')
    shutil.copytree(src, dst, dirs_exist_ok=True, ignore=ignore)
    print(f'mirrored {src} -> {dst}')

## 4b · Persistent processed-dataset cache — build once, reuse every session

The slow part of every run is turning the raw ZuCo `.mat` files into processed tensors (band power, normalisation, imputation). That result depends only on the **dataset config**, not on the model/objective, so it is identical across experiments, held-out subjects and sessions — yet it used to be rebuilt on every single run.

Now `zte-run --data-cache <dir>` stores the processed bundle in a **shared, content-addressed** store: the first run builds it, every later run (any experiment, any session) loads it and **skips the `.mat` load + processing entirely**. The cell below keeps that store on Drive under a general path (`Sharables/ZTE/prepared/`, *not* date-stamped — it is reusable forever) and mirrors it to a fast **local** copy that runs read from:

- `restore_prepared_cache()` — pulls previously-built bundles Drive → local at session start, so runs start warm.
- `backup_prepared_cache()` — pushes newly-built bundles local → Drive. Each bundle is content-addressed and immutable, so both directions only copy what is genuinely new (cheap no-ops otherwise).

Every run cell below passes `--data-cache "{PREPARED_LOCAL}"` (and the sweep scripts in Sections 6/6b read `$DATA_CACHE`), so each distinct dataset is processed once and never again.

In [ ]:
# Persistent PROCESSED-dataset cache: build once, reuse across experiments & sessions (skips the .mat
# load + processing). The bundle is content-addressed by the dataset config, so different
# representations (band-power vs raw, different bands/windows, ...) get distinct entries and never collide.
import os

PREPARED_DRIVE = f'{ZTE_DRIVE}/prepared'  # GENERAL path (NOT date-stamped) — the processed bundle is reusable forever
PREPARED_LOCAL = 'res/cache/prepared'  # runs read from here (local = fast); passed as --data-cache
os.environ['DATA_CACHE'] = PREPARED_LOCAL  # exposes it to the sweep scripts (Sections 6 / 6b) too


def _sync_bundles(src: str, dst: str, label: str) -> None:
    """Copy content-addressed bundle subfolders src->dst, skipping any already present (they are immutable)."""
    s = pathlib.Path(src)
    if not s.is_dir():
        print(f'no processed-dataset cache at {src} yet — {label}')
        return
    d = pathlib.Path(dst)
    d.mkdir(parents=True, exist_ok=True)
    n = 0
    for sub in s.iterdir():
        if sub.is_dir() and not (d / sub.name).exists():
            shutil.copytree(sub, d / sub.name)
            n += 1
    print(f'{label}: {n} new bundle(s)  {src} -> {dst}')


def restore_prepared_cache() -> None:
    """Pull previously-built processed bundles Drive -> local so this session starts warm."""
    _sync_bundles(
        PREPARED_DRIVE, PREPARED_LOCAL, 'restored (the first run builds any still missing)'
    )


def backup_prepared_cache() -> None:
    """Push newly-built processed bundles local -> Drive so future sessions skip re-processing."""
    _sync_bundles(PREPARED_LOCAL, PREPARED_DRIVE, 'backed up')


restore_prepared_cache()  # start warm: reuse anything already processed in a previous session

## 5 · Run the SOTA experiments — one spotlight run, or a resumable series

`experiments/flagship/clip_e5_bandpower.yaml` implements the full road-to-state-of-the-art recipe (`docs/METHODS.md`), all grounded in the neuroscience/engineering literature and each provable in isolation (Section 6c):

- **Fix the retrieval geometry** — `all_but_top` (Mu & Viswanath 2018) + `csls_neighbors` (Conneau 2018) strip the anisotropy/hubness that made cross-subject retrieval sit *below* chance on a healthy space. The single highest-leverage change.
- **Rebalance + ramp the subject adversary** — weight cut `1.0 → 0.1` (the EEG invariance literature uses 0.03–0.05; Özdenizci 2020) and its gradient-reversal strength ramps from zero (Ganin 2016), so invariance no longer erodes the content it should preserve.
- **Sharpen the contrastive loss** — `alignment_weight` (the missing half of align+uniformity, Wang & Isola 2020), `tau_plus` debiased InfoNCE (Chuang 2020), and a `data2vec_aux_weight` frozen-target head that fills the factored nuisance dims.
- **Exact montage + spatial model** — real `channel,x,y,z` geometry feeds the spherical-harmonic encoder; the `exp7` A/B swaps in learned spatial attention + FiLM subject conditioning (Défossez 2023).
- **Bulletproof evaluation** — the retrieval verdict is gated on the permutation null; seen-vs-novel and frequency-matched retrieval and **rank-percentile** (the pre-registered success metric) are all reported.

**The two flagship configs to run and compare:**
- `experiments/flagship/clip_e5_bandpower.yaml` — the geometry-fixed spherical-harmonic SOTA (the headline).
- `experiments/flagship/clip_e5_meaning.yaml` — the spatial-attention + FiLM A/B.

**Turn-key ingredients (no manual prereq cells).** The exact electrode montage and the word-meaning target used to be hand-built (export a montage CSV, download GloVe, hand-edit the YAML). They are now **one flag each on `zte-run`**, and every run cell below sets them:
- `--spatial {exact,attention,approx,off}` — builds + wires the montage (exact per-channel geometry **and** regions). `exact` needs `mne` (installed by Section 1's `uv sync --all-groups`); without it, it falls back to the approximate cap.
- `--meaning {static,contextual,hash}` — builds + wires the distillation target: `static` = vocab-restricted GloVe, `contextual` = per-occurrence frozen-encoder (e.g. BERT mid-layer). `contextual` needs `transformers` (also from `--all-groups`).

Each artifact is **built once and cached under `res/`** (montage → `res/montage_gsn105.csv`, GloVe → `res/vectors/glove.300d.txt`, and the expensive contextual-BERT matrix → `res/cache/meaning/`), then reused across every held-out subject, ablation arm, and re-run — so nothing expensive is recomputed. The same flags work on `zte-ablate generate`, which now also sweeps a **grid** (repeat `--knob/--values` for the Cartesian product).

**How this section is organised.** Everything writes straight to Drive and is **`--resume`-safe** — a complete run is skipped *instantly*, an interrupted one continues from its last checkpoint.

- **5-iii** — **one spotlight experiment**: a single config × one held-out subject. Start here.
- **5-iv** — a **series of spotlight experiments**: the two flagship configs in a resumable loop.
- **5-v** — **compare** the series on the held-out north-star.
- **5-vi** — the **CLIP sentence-alignment A/B** (the content-gap pivot).

For the exhaustive, **multi-hour** runs use **Section 6** (full LOSO over *every* subject) and **Section 6b** (the full bias-controlled suite). Prove each lever in isolation in **Section 6c**, and benchmark the four objectives in **Section 7**.

In [ ]:
# 5-iii · Run ONE spotlight experiment — a single config × one held-out subject.
CONFIG: str = 'experiments/flagship/clip_e5_bandpower.yaml'  # the champion (only recipe that beat chance on real ZuCo); or clip_e5_raw.yaml / clip_e5_meaning.yaml
HOLDOUT: str = 'ZAB'  # the held-out 'new brain' for this run
SPATIAL: str = 'exact'  # exact ZuCo-105 montage (spatial encoding + regions); or approx / attention / off / keep
MEANING: str = 'keep'  # vocab-restricted GloVe target; or contextual (per-occurrence BERT mid-layer) / hash / keep

# Real data — trains + fully evaluates and writes EVERYTHING straight to Drive. --spatial/--meaning build
# the montage + meaning target ONCE (cached under res/, reused by every later run); --data-cache reuses the
# processed dataset bundle so the .mat load + processing is skipped. --resume is idempotent.
!uv run zte-run --config {CONFIG} --root "{DATA_DIR}" --spatial {SPATIAL} --meaning {MEANING} \
    --data-cache "{PREPARED_LOCAL}" --loso-holdout {HOLDOUT} --out-root "{DRIVE_DIR}/experiments" --resume
backup_prepared_cache()  # persist the processed bundle to Drive so future sessions skip re-processing

# (Synthetic smoke — mechanics only, no data, stays local. Uncomment to sanity-check the whole stack.)
# !uv run zte-run --config {CONFIG} --synthetic --epochs 3 --spatial {SPATIAL} --meaning {MEANING} \
#     --loso-holdout {HOLDOUT} --out-root res/experiments

# Resolve the run dir from the config's own run_name + the LOSO suffix, then show the honest headline:
import yaml

_run_name = yaml.safe_load(open(CONFIG))['run_name']
RUN_DIR = f'{DRIVE_DIR}/experiments/{_run_name}_lo{HOLDOUT}'
rep = pathlib.Path(f'{RUN_DIR}/evaluation/report.md')
if rep.exists():
    t = rep.read_text()
    print(t[t.find('## Scoreboard') : t.find('## Verdict')] or t[:1500])
print('run dir ->', RUN_DIR)
print(
    'interactive held-out scoreboard ->',
    f'{RUN_DIR}/evaluation/interactive/held_out_scoreboard.html',
)

In [ ]:
# 5-iv · Run a SERIES of spotlight experiments (resumable, all backed up to Drive).
# Each (config, held-out subject) is trained + fully evaluated and written straight to Drive. --resume
# skips any run already complete (instant) and continues an interrupted one, so re-running this cell
# never redoes finished work. Comment out an entry once it is done (or just leave it — it is skipped).
SPATIAL: str = 'exact'  # exact ZuCo-105 montage for every run; or approx / attention / off / keep
MEANING: str = (
    'keep'  # vocab-restricted GloVe target; or contextual (per-occurrence BERT) / hash / keep
)
SPOTLIGHT = [
    (
        'experiments/flagship/clip_e5_bandpower.yaml',
        'ZAB',
    ),  # geometry-fixed spherical-harmonic SOTA (the headline)
    ('experiments/flagship/clip_e5_meaning.yaml', 'ZAB'),  # spatial-attention + FiLM A/B
    # ('experiments/flagship/clip_e5_bandpower.yaml', 'ZDM'),                # add more (config, held-out subject) pairs here
]

import pathlib

import yaml

# Montage, meaning target AND the processed dataset are built once by the first run and reused from cache.
for _cfg, _holdout in SPOTLIGHT:
    _rn = yaml.safe_load(open(_cfg))['run_name']
    _dir = f'{DRIVE_DIR}/experiments/{_rn}_lo{_holdout}'
    _done = pathlib.Path(f'{_dir}/evaluation/report.md').exists()
    print(
        f'=== {_rn} · held-out {_holdout} · {"already complete (skipped)" if _done else "training/resuming"} -> {_dir}'
    )
    !uv run zte-run --config {_cfg} --root "{DATA_DIR}" --spatial {SPATIAL} --meaning {MEANING} --data-cache "{PREPARED_LOCAL}" --loso-holdout {_holdout} --out-root "{DRIVE_DIR}/experiments" --resume
backup_prepared_cache()  # persist the processed bundle(s) to Drive for future sessions
print(
    'done — every spotlight run (checkpoints + eval + figures + interactive scoreboard) is on Drive under',
    f'{DRIVE_DIR}/experiments',
)

### 5-v · CLIP — sentence-level EEG↔text semantic alignment (the content-gap pivot)

The strongest lever for the **content** gap: instead of self-supervised skip-gram, align each sentence's
pooled EEG vector to a **frozen sentence embedding of its ground-truth text** with a symmetric InfoNCE
loss (CLIP; Radford 2021; Défossez 2023), plus **semantic-hard negatives** (surface-similar, meaning-distinct).
VICReg + the invariance levers stay on as auxiliaries. Full write-up + exact tensor shapes:
`docs/CLIP_ALIGNMENT.md`.

Runs the **A/B over both text encoders** — `clip_e5_bandpower` (E5 sentence model) vs `clip_qwen_bandpower` (Qwen,
mean-pooled) — on held-out ZAB, so you can see which target the EEG aligns to best on the held-out
north-star (Section 5-v compares them). Needs the frozen encoders — installed by Section 1's
`uv sync --all-groups`; without them it falls back to a hash target with a warning and still runs.
`clip_e5_raw` is the staged raw-conformer (TSConv, ~700 ms window) — run it after the word-pool A/B
validates the objective.

In [ ]:
# 5-vi · CLIP sentence-alignment A/B (E5 vs Qwen), held out on ZAB. Resumable + backed up to Drive.
# Same --resume semantics as 5-iv: a completed run is skipped instantly, an interrupted one continues.
# CLIP aligns EEG to a frozen TEXT encoder (text_source), so --meaning is not used here; --spatial exact
# still builds + wires the shared montage, and --data-cache reuses the processed dataset (all cached).
SPATIAL: str = 'exact'  # exact ZuCo-105 montage for every arm; or approx / attention / off / keep
CLIP_AB = [
    (
        'experiments/flagship/clip_e5_bandpower.yaml',
        'ZAB',
    ),  # E5 sentence-embedding target (word-pool encoder)
    (
        'experiments/benchmark/clip_qwen_bandpower.yaml',
        'ZAB',
    ),  # Qwen mean-pooled target (word-pool encoder)
    (
        'experiments/flagship/clip_e5_raw.yaml',
        'ZAB',
    ),  # Stage 2: raw-conformer TSConv encoder (~700 ms window)
]

import pathlib

import yaml

for _cfg, _holdout in CLIP_AB:
    _rn = yaml.safe_load(open(_cfg))['run_name']
    _dir = f'{DRIVE_DIR}/experiments/{_rn}_lo{_holdout}'
    _done = pathlib.Path(f'{_dir}/evaluation/report.md').exists()
    print(
        f'=== {_rn} · held-out {_holdout} · {"already complete (skipped)" if _done else "training/resuming"}'
    )
    !uv run zte-run --config {_cfg} --root "{DATA_DIR}" --spatial {SPATIAL} --data-cache "{PREPARED_LOCAL}" --loso-holdout {_holdout} --out-root "{DRIVE_DIR}/experiments" --resume
backup_prepared_cache()  # persist the processed bundle(s) to Drive for future sessions
print('done — CLIP A/B runs on Drive. Re-run 5-v to compare E5 vs Qwen vs the geometry-only SOTA.')

In [ ]:
# 5-vi · Compare the spotlight runs — see how each improvement performs on the held-out north-star
# (cross-subject retrieval, rank-percentile, content-lift-over-raw), pass/fail on the four honesty
# checks, side by side. Builds the cross-run dashboard from Drive and renders it inline.
!uv run zte-compare --experiments "{DRIVE_DIR}/experiments" --out "{DRIVE_DIR}/experiments/COMPARE.html"

# A compact headline table across the runs (reads each run's metrics.json — no ZTE import needed):
import pandas as pd
from IPython.display import HTML, display  # type: ignore[import-untyped]

_rows = []
for _mf in sorted(pathlib.Path(f'{DRIVE_DIR}/experiments').glob('*/evaluation/metrics.json')):
    _m = json.load(open(_mf))
    _sb = (_m.get('scoreboard') or {}).get('cross_subject_holdout_retrieval') or {}
    _rows.append(
        {
            'run': _mf.parent.parent.name,
            'retrieval_top1': _m.get('sentence_retrieval', {}).get('top1'),
            'chance_top1': _m.get('sentence_retrieval', {}).get('chance_top1'),
            'rank_percentile': _m.get('sentence_retrieval', {}).get('rank_percentile'),
            'held_out_retrieval_top1': _sb.get('top1'),
            'held_out_lift_top1': _sb.get('lift_top1'),
            'eff_rank_ratio': _m.get('embedding_health', {}).get('effective_rank_ratio'),
            'perm_above_chance': _m.get('verdict', {}).get('retrieval_above_chance_perm'),
        }
    )
if _rows:
    display(pd.DataFrame(_rows).set_index('run'))
display(HTML(open(f'{DRIVE_DIR}/experiments/COMPARE.html', encoding='utf-8').read()))

## 6 · Full LOSO on the SOTA config — the “new brain” sweep (multi-hour, resumable)

Once a spotlight run (Section 5) looks good, rotate the held-out subject over the **whole cohort** with the same config, turning one data point into a **trend** (`COMPARE.html`). `FULL_CFG` selects it (defaults to the SOTA config). This is the exhaustive, **multi-hour** away-game — one model per held-out subject. Fast local training + a live per-epoch checkpoint mirror to Drive; every run is `--resume`-safe (finished subjects skip instantly, an interrupted one continues), so a dropped runtime never loses work — just re-run. The **real** sweep runs by default below; the synthetic dry-run is commented out.

In [ ]:
# === REAL full LOSO sweep on the SOTA config — rotate the held-out subject over the WHOLE cohort ===
# One config × every subject = the exhaustive 'does it generalise to any stranger' trend (COMPARE.html).
# MULTI-HOUR. Fast local training + a live per-epoch checkpoint mirror to Drive; --resume (built in)
# skips finished subjects and continues an interrupted one, so it is safe to re-run.
# SPATIAL/MEANING/DATA_CACHE provision the montage, meaning target AND processed dataset ONCE and reuse
# them from cache across all 12 held-out subjects (all subject-independent) — nothing is recomputed per subject.
!SPATIAL=exact MEANING=static DATA_CACHE="{PREPARED_LOCAL}" FULL_CFG=experiments/flagship/clip_e5_bandpower.yaml DRIVE_BACKUP="{DRIVE_DIR}/loso" bash scripts/run_loso.sh "{DATA_DIR}"
mirror_to_drive(
    'res/experiments/loso', 'loso'
)  # push eval/figures/interactive to Drive once training completes
backup_prepared_cache()  # persist the processed bundle to Drive for future sessions

# (Synthetic dry-run of the whole sweep — mechanics only, CPU, stays local. Uncomment to sanity-check.)
# !SMOKE=1 FULL_CFG=experiments/flagship/clip_e5_bandpower.yaml bash scripts/run_loso.sh

# Extras:  FULL_CFG=experiments/flagship/clip_e5_meaning.yaml (the spatial-attn A/B) ·
#          SUBJECTS="ZAB ZDM" (restrict the held-out set)  ·  OUT_ROOT="{DRIVE_DIR}/loso" (write straight to Drive)

## 6b · Full experiment suite — the fixed-seed driver
`scripts/run_suite.sh` runs the tiered suite at a **single fixed seed (42)**, each arm held out on ZAB. `STUDIES` picks what runs (default `audit flagship controls`): `audit` = the model-free confound report, `flagship` = the three CLIP arms (`clip_e5_bandpower`, `clip_e5_raw`, `clip_e5_meaning`), `controls` = the skip-gram baseline and the Qwen text-encoder arm. `benchmark`, `ablate` and `loso` add the objective sweep, the one-knob studies and the full 12-subject sweep. Every run is `--resume`-safe, so an interrupted suite continues instantly where it stopped.

**Best of both worlds (the default).** Train on the **fast local disk** while `DRIVE_BACKUP` mirrors each run's `best.pt`/`last.pt` to Drive **every epoch** — so trainable progress is never lost if the runtime dies — and then `backup_to_drive()` writes **everything else** (evaluation, figures, interactive HTML) to Drive once training completes. You get fast training I/O *and* full persistence. Interrupted? Set `RESUME_DATE` (Section 4), `restore_from_drive()`, and re-run: finished runs skip, the interrupted one continues, restored runs re-evaluate (cheap), and the final `backup_to_drive()` re-syncs. The kept-open alternative — `OUT_ROOT=…/experiments` on Drive — writes *everything* to Drive live (simplest resume, no restore) but pays ~20–30 s/epoch on big raw-conformer checkpoints. `SMOKE=1` = fast synthetic dry-run (stays local). Uncomment `study_loso_sweep` inside the script for the full 12-subject leave-one-out sweep on the flagship.

In [ ]:
# === REAL full suite on Colab — the fixed-seed (42) bias-controlled study set (MULTI-HOUR) ===
# Trains on the FAST local disk while DRIVE_BACKUP mirrors each run's best/last.pt to Drive EVERY EPOCH
# (trainable progress is never lost if the runtime dies), then backup_to_drive() pushes EVERYTHING —
# eval, figures, interactive, benchmark — to Drive once training completes. --resume (built in) skips
# finished runs instantly and continues an interrupted one, so re-running never redoes finished work.
# SPATIAL=exact provisions the shared montage once; DATA_CACHE reuses the processed dataset bundle across
# every run. The suite mixes meaning-distillation and CLIP configs, so MEANING is left per-config.
!SPATIAL=exact DATA_CACHE="{PREPARED_LOCAL}" DRIVE_BACKUP="{DRIVE_DIR}/experiments" BENCH_ROOT="{DRIVE_DIR}/benchmark" bash scripts/run_suite.sh "{DATA_DIR}"
backup_to_drive(
    note='full suite (seed 42)'
)  # writes res/experiments/* (eval/figures/interactive) to Drive
backup_prepared_cache()  # persist the processed bundle(s) to Drive for future sessions

# (Synthetic smoke of the ENTIRE suite — mechanics only, seconds, stays local. Uncomment to sanity-check.)
# !SMOKE=1 bash scripts/run_suite.sh

# --- Option (kept open): write EVERYTHING straight to Drive as produced (simplest resume, slower ckpt I/O) ---
# !SPATIAL=exact DATA_CACHE="{PREPARED_LOCAL}" OUT_ROOT="{DRIVE_DIR}/experiments" BENCH_ROOT="{DRIVE_DIR}/benchmark" bash scripts/run_suite.sh "{DATA_DIR}"

# RESUME after a runtime reset (DRIVE_BACKUP flow): set RESUME_DATE (Section 4), then restore_from_drive()
# to pull mirrored checkpoints back, and re-run the command above (--resume skips done) + backup_to_drive().

## 6c · Prove each lever — single-variable ablation (`zte-ablate`)
The scoreboard only becomes *proof* when each lever is tested in isolation. `zte-ablate generate` writes a config sweep that changes **exactly one knob**; run both arms, then `zte-ablate diff` reports that knob's contribution to the held-out LOSO north-star — everything else identical. This is the discipline the original reports could only apply to VICReg.

In [ ]:
# Prove each lever in isolation against the held-out LOSO north-star. `zte-ablate` drives ANY
# dotted section.field with zero code change, so every lever is a clean single-variable A/B:
#   objective.subject_adversary_weight  (0 / 0.1 / 0.3)  <- the one most likely to unblock retrieval
#   objective.all_but_top               (0 / 1)          <- the geometry fix (anti-hubness)
#   objective.csls_neighbors            (0 / 10)         <- CSLS retrieval correction
#   objective.alignment_weight          (0 / 0.1)        <- align+uniformity's missing half
#   objective.tau_plus                  (0 / 0.1)        <- debiased contrastive
#   objective.data2vec_aux_weight       (0 / 0.5)        <- collapse-insurance / fill nuisance dims
#   model.spatial_encoding  (spherical_harmonics / spatial_attention)   model.subject_film (0 / 1)
KNOB = 'objective.subject_adversary_weight'  # cut the adversary to unblock content — the highest-leverage sweep

# 1) generate the one-knob sweep from the SOTA config
!uv run zte-ablate generate --config experiments/flagship/clip_e5_bandpower.yaml --knob {KNOB} --values 0,0.1,0.3 --out-dir res/ablate_configs

# 1-grid) Try COMBINATIONS instead: repeat --knob/--values for the Cartesian product, and provision the ingredients ONCE
# with --spatial/--meaning so every arm shares the exact montage. E.g. spatial encoding × meaning target (3 × 2 = 6 configs)
# reveals whether exact geometry only helps once meaning is on:
# !uv run zte-ablate generate --config experiments/flagship/clip_e5_bandpower.yaml --spatial exact \
#     --knob model.spatial_encoding --values none,spherical_harmonics,spatial_attention \
#     --knob objective.meaning_contextual --values None,bert-base-uncased --out-dir res/ablate_configs

# 2) run each arm on real data -> Drive. --resume skips arms that are already complete (re-run freely).
for cfg in sorted(glob.glob('res/ablate_configs/*.yaml')):
    !uv run zte-run --config {cfg} --root "{DATA_DIR}" --loso-holdout ZAB --out-root "{DRIVE_DIR}/ablate" --resume

# 3) diff the scoreboards -> the knob's isolated contribution on the held-out north-star (retrieval + rank-percentile)
metrics = sorted(glob.glob(f'{DRIVE_DIR}/ablate/*/evaluation/metrics.json'))
if len(metrics) >= 2:
    !uv run zte-ablate diff --knob {KNOB} --baseline {metrics[0]} --variant {metrics[-1]}

## 7 · Benchmark objectives on real ZuCo (the flagship sweep)
A reproducible grid — **all four self-supervised objectives (skip-gram / CBOW / masked / CPC) x RoPE x {EEG-only vs +eye-tracking}** at the fixed **seed 42** — trained and fully evaluated on real ZuCo. It answers ZTE's two load-bearing questions in one table: *which objective best encodes thought*, and *how much of any score is the eye-tracking confound* (EEG-only is the honest headline). Results are ranked by `subject_transfer_lift` (does the same word transfer across brains, above chance) and written **straight to Drive** so the multi-hour sweep survives a runtime reset. A one-line **synthetic smoke** is included (commented) for a seconds-long mechanics check.

In [ ]:
import pandas as pd

# Where the benchmark is written. Drive = persisted across runtime resets (needs Section 4 mounted).
BENCH_OUT: str = f'{DRIVE_DIR}/benchmark' if 'DRIVE_DIR' in dir() else 'res/benchmark'

# ===== REAL benchmark on ZuCo — the flagship comparison (runs by default) =======================
# Grid = all 4 self-supervised objectives x RoPE x {EEG-only vs +eye-tracking}, fixed seed 42, on
# real ZuCo (SR + NR). It settles the two questions that decide whether ZTE is a *standard*:
#   (1) which objective best encodes thought?   (2) how much does the eye-tracking confound inflate it?
# 8 models are trained AND fully evaluated (hours on a Colab GPU) and written straight to Drive, so a
# dropped runtime never loses the sweep. Each cell also saves its resolved config.yaml (reproducible).
!uv run zte-benchmark --root "{DATA_DIR}" --tasks SR,NR --objectives skipgram,cbow,masked,cpc \
    --pos-encodings rope --eye-tracking both --seeds 42 --epochs 20 --batch-size 128 --device auto --out "{BENCH_OUT}"

# ----- Fast synthetic smoke instead (seconds; set BENCH_OUT = "res/benchmark" above, then run) ---
# !uv run zte-benchmark --synthetic --objectives skipgram,masked --pos-encodings rope --eye-tracking off --seeds 42 --epochs 3 --out res/benchmark

# ----- Fuller study (optional): widen any axis --------------------------------------------------
# ...  --pos-encodings rope,sinusoidal,alibi    --tasks SR,NR,TSR    --seeds 42,43,44

# ----- Rank the results (CSV is pre-sorted by subject_transfer_lift — the cross-subject north-star)
bench_df = pd.read_csv(f'{BENCH_OUT}/benchmark.csv')
_head = [
    'objective',
    'eye_tracking',
    'sent_retrieval_top1',
    'subject_transfer_lift',
    'eff_rank_ratio',
    'beats_noise',
]
bench_df[[c for c in _head if c in bench_df.columns]]

In [ ]:
# Compare objectives visually: cross-subject transfer & retrieval, EEG-only vs +eye-tracking.
import matplotlib.pyplot as plt

need = {'objective', 'eye_tracking', 'subject_transfer_lift', 'sent_retrieval_top1'}
if need <= set(bench_df.columns):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    for ax, metric in zip(axes, ['subject_transfer_lift', 'sent_retrieval_top1']):
        bench_df.pivot_table(index='objective', columns='eye_tracking', values=metric).plot.bar(
            ax=ax, rot=0
        )
        ax.set(title=metric, xlabel='objective', ylabel=metric)
        ax.grid(axis='y', alpha=0.3)
        ax.legend(title='eye_tracking')
    fig.suptitle('ZuCo benchmark — objective x eye-tracking (seed 42)')
    plt.tight_layout()
    plt.show()
else:
    print('Run the benchmark cell first (or it was a narrower grid).')

# Persist the benchmark to Drive alongside everything else (safe if you mounted Drive).
if 'backup_to_drive' in dir():
    backup_to_drive(note='benchmark: objectives x eye-tracking (seed 42)')

## 8 · Visualise & interact (HTML)
Build the interactive **Thought-Space Explorer** + **Neuron Atlas** for a run, and the **comparison dashboard** across all runs. The dashboard is small enough to render inline; the 5 MB explorers are best downloaded (Section 9) and opened locally for full 3-D interaction.

In [ ]:
!uv run zte-visualize --run res/experiments/exp8_clip_e5_loZAB --kind both
!uv run zte-compare --experiments res/experiments --out res/experiments/COMPARE.html
from IPython.display import HTML  # type: ignore[import-untyped]

HTML(filename='res/experiments/COMPARE.html')  # the scorecard + best-run dashboard, inline

In [ ]:
# The NEW interactive held-out scoreboard — the blocking "new brain" numbers as animated gauges
# (effective rank, anisotropy, content lift over raw, identity leak, cross-subject retrieval vs its
# chance line, and rank-percentile), each against its named reference. Written automatically by every
# eval to <run>/evaluation/interactive/held_out_scoreboard.html. Point RUN at the SOTA run and display:
import glob

from IPython.display import HTML  # type: ignore[import-untyped]

_boards = (
    sorted(glob.glob('res/experiments/*/evaluation/interactive/held_out_scoreboard.html'))
    + sorted(
        glob.glob(f'{DRIVE_DIR}/experiments/*/evaluation/interactive/held_out_scoreboard.html')
    )
    if 'DRIVE_DIR' in dir()
    else sorted(glob.glob('res/experiments/*/evaluation/interactive/held_out_scoreboard.html'))
)
if _boards:
    display(HTML(open(_boards[-1], encoding='utf-8').read()))  # newest run's dashboard, inline
else:
    print('No held-out scoreboard yet — train + evaluate a run (Section 5) first.')

## 8b · Explore the training you just ran — tables, charts & images
A quick, self-contained look at your runs **without leaving the notebook** — reads each run's `manifest.json` / `metrics.json` / figures directly (no ZTE import needed, so it works in the plain Colab kernel). You get a **scorecard DataFrame** across all runs, an interactive **run picker** (a Colab dropdown auto-populated from `res/experiments/`) that shows the selected run's **training curves** and key **evaluation figures** inline, a **comparison bar chart**, and a **metrics deep-dive** table for the selected run. (The PCA-by-subject thumbnail in the Section 8 dashboard now embeds itself as a data-URI, so it renders inline on Colab too.)

In [ ]:
# Scorecard: every run's headline metrics in one tidy table (reads manifest.json).
import pandas as pd

rows = []
for mf in sorted(glob.glob('res/experiments/*/manifest.json')):
    m = json.load(open(mf))
    ev = m.get('evaluation') or {}
    verdict = ev.get('verdict') or {}
    passed = sum(1 for v in verdict.values() if v is True) if isinstance(verdict, dict) else None
    rows.append(
        {
            'run': pathlib.Path(mf).parent.name,
            'train_loss': m.get('final_train_loss'),
            'retrieval_top1': ev.get('sentence_retrieval_top1'),
            'subject_transfer_top1': ev.get('subject_transfer_top1'),
            'eff_rank_ratio': ev.get('effective_rank_ratio'),
            'checks_passed': passed,
        }
    )
scorecard = pd.DataFrame(rows).sort_values('retrieval_top1', ascending=False, na_position='last')
scorecard.reset_index(drop=True)

In [ ]:
# Pick a run from a dropdown and view its training + evaluation figures (interactive, Colab widgets).
import glob
import os

from IPython.display import Image, Markdown, display  # type: ignore[import-untyped]

RUNS = sorted(
    os.path.basename(os.path.dirname(p)) for p in glob.glob('res/experiments/*/manifest.json')
)
FIGURES = [
    ('Training curves (loss / lr)', 'checkpoints/training_curves.png'),
    # --- the blocking story: geometry & cross-subject retrieval ---
    (
        'Geometry before vs after (anti-cone / anti-hubness fix)',
        'evaluation/figures/geometry_before_after.png',
    ),
    (
        'Retrieval rank distribution (the pre-registered success metric)',
        'evaluation/figures/retrieval_rank_distribution.png',
    ),
    (
        'Cross-subject sentence retrieval (Top-K vs chance)',
        'evaluation/figures/retrieval_sentence.png',
    ),
    (
        'Cross-subject centroid similarity (hubness/identity diagnostic)',
        'evaluation/figures/subject_similarity.png',
    ),
    # --- what the space encodes: who vs what, and where on the scalp ---
    (
        'Variance budget — who (subject) vs what (word)',
        'evaluation/figures/variance_budget_pie.png',
    ),
    (
        'Neuron selectivity — top dimensions × attributes',
        'evaluation/figures/neuron_selectivity.png',
    ),
    (
        'Scalp topomap — electrodes carrying lexical-frequency info',
        'evaluation/figures/scalp_topomap.png',
    ),
    ('PCA of embeddings by subject', 'evaluation/figures/pca_by_subject.png'),
    ('Embedding health (per-dim std + PCA spectrum)', 'evaluation/figures/embedding_health.png'),
    (
        'Linear-probe comparison (ZTE vs raw vs noise vs phase-shuffled)',
        'evaluation/figures/probe_linear.png',
    ),
    ('Scalp-region importance heatmap', 'evaluation/figures/region_importance.png'),
]


def show_run_figures(run: str) -> None:
    """Render one run's figures inline (embeds the PNGs, so they show on Colab too)."""
    base = f'res/experiments/{run}'
    display(Markdown(f'### `{run}` — training & evaluation figures'))
    for caption, rel in FIGURES:
        path = f'{base}/{rel}'
        if os.path.exists(path):
            display(Markdown(f'**{caption}** — `{rel}`'))
            display(Image(filename=path))
        else:
            display(Markdown(f'_missing: {rel}_'))


run_selector = None
if not RUNS:
    print('No runs yet — train one first (Section 5), then re-run this cell.')
else:
    try:
        import ipywidgets as widgets  # type: ignore[import-untyped]

        run_selector = widgets.Dropdown(options=RUNS, value=RUNS[0], description='Run:')
        # Reactive: changing the dropdown re-renders below; run_selector.value = current pick.
        widgets.interact(show_run_figures, run=run_selector)
    except ImportError:  # no widgets (e.g. plain terminal) -> just show the first run
        show_run_figures(RUNS[0])

In [ ]:
# Comparison bar chart across runs (matplotlib, from the scorecard).
import matplotlib.pyplot as plt

comp = scorecard.dropna(subset=['retrieval_top1']).set_index('run')
if len(comp):
    ax = comp[['retrieval_top1', 'eff_rank_ratio']].plot.bar(
        figsize=(1.6 * len(comp) + 3, 4), rot=15
    )
    ax.set(title='Runs compared — retrieval Top-1 & effective-rank ratio', ylabel='value')
    ax.grid(axis='y', alpha=0.3)
    ax.legend(title='metric')
    plt.tight_layout()
    plt.show()
else:
    print('No evaluated runs yet — run a training cell first.')

In [ ]:
# Deep-dive: the full metrics.json for the SELECTED run (re-run after changing the dropdown above).
run = run_selector.value if run_selector is not None else (RUNS[0] if RUNS else None)
if run:
    metrics_path = f'res/experiments/{run}/evaluation/metrics.json'
    if os.path.exists(metrics_path):
        metrics = json.load(open(metrics_path))
        flat = pd.json_normalize(metrics, sep='.').T.rename(columns={0: 'value'})
        display(Markdown(f'### `{run}` — full evaluation metrics'))
        display(flat)
    else:
        print('No metrics.json — evaluation may have been skipped for', run)
else:
    print('No run available — train one first (Section 5).')

## 9 · Back up to Drive — lightweight archive + full snapshot
Two complementary saves, both landing under `Sharables/ZTE/{RUN_DATE}/` (permanent & shareable). **Smoke / `--synthetic` runs are skipped by default** — only real runs reach Drive (a session with only smoke runs is a friendly no-op; pass `include_synthetic=True` to force).

- **`backup_to_drive()`** — frequent, cheap. A provenance-stamped **best-only zip** (git commit + versions + each run's `config.yaml` + metrics) *and* a browsable mirror of reports/figures/3-D explorers. Call it after each **real** training step.
- **`snapshot_to_drive()`** — the **continue-locally** bundle. Zips the whole working state — `experiments` (real runs) + `cache` (the dataset cache!) + `benchmark` + `explorer` — into one file. Download it, `zte-pack unpack … --dest res` on your machine, and keep exploring / training **without paying for more GPU time**.

Both carry `PROVENANCE.json`/`PROVENANCE.md`. For long real-data runs, also write straight to Drive via `OUT_ROOT` (Sections 6/6b) so runs persist the instant they finish.

In [ ]:
!uv run zte-pack list

In [ ]:
# Back up EVERYTHING to Drive: all [best] runs as a provenance-stamped zip + a browsable mirror.
backup_to_drive(note=f'session {RUN_DATE}: all best runs')

# What you now have on Drive (permanent, shareable):
#   {DRIVE_DIR}/archives/zte_<date>_<time>.zip   restorable bundle (best.pt + config + eval + PROVENANCE.json/md)
#   {DRIVE_DIR}/experiments/<run>/               browsable reports, figures & 3-D explorers per run
#   {DRIVE_DIR}/benchmark/                        benchmark tables

# Restore later (new Colab session, or your locally) from the newest archive:
# !ls -t "{DRIVE_DIR}/archives"/*.zip | head -1
# !uv run zte-pack unpack "{DRIVE_DIR}/archives/<the>.zip" --dest res/experiments

# Download a single archive to your browser instead:
# from google.colab import files; files.download(f"{DRIVE_DIR}/archives/...zip")

# Free Colab space once it is safely on Drive (add --move to the zip, or):
# !uv run zte-pack clean experiments benchmark --yes

In [ ]:
# FULL snapshot -> Drive: experiments + cache + benchmark + explorer in ONE zip.
# Download this single file and keep exploring / continue training LOCALLY — no GPU needed
# (the dataset cache is bundled, so a local session doesn't re-prepare the data).
snapshot_to_drive(note=f'full working-state snapshot {RUN_DATE}')

# Pick specific subtrees only (e.g. skip the big cache):
# snapshot_to_drive(targets=["experiments", "benchmark", "explorer"])

# Restore locally (or in a fresh Colab) — recreates res/experiments, res/cache, res/benchmark, res/explorer:
# !uv run zte-pack unpack "{DRIVE_DIR}/archives/<the_snapshot>.zip" --dest res

## 10 · Run it locally (inference)
Grab the newest archive from your Drive (`Sharables/ZTE/<date>/archives/`) — it carries `best.pt`, each run's `config.yaml`, the evaluation, and `PROVENANCE.json`/`PROVENANCE.md`. Then, in a terminal on your machine (Apple-silicon MPS is picked up automatically):

```sh
uv sync --group all
uv run zte-pack unpack ~/Downloads/zte_<date>_<time>.zip --dest res/experiments   # or straight from a synced Drive path
cat res/experiments/PROVENANCE.md            # git commit + versions + per-run metrics (how it was produced)

# Re-open the interactive explorer for a run:
open res/experiments/colab_exp6/evaluation/interactive/thought_space_explorer.html

# Extract embeddings from the trained checkpoint (best.pt is enough — shapes + normaliser are baked in):
uv run zte-extract --ckpt res/experiments/colab_exp6/checkpoints/best.pt --root "/path/to/ZuCo Dataset" --out res/embeddings/exp6.npz

# Or re-run just the evaluation / comparison locally:
uv run zte-compare --experiments res/experiments
```

To reproduce a run exactly: `git checkout <commit from PROVENANCE.md>`, `uv sync --group all`, then `uv run zte-run --config res/experiments/<run>/config.yaml --root "/path/to/ZuCo Dataset" --name <run>`.

In [ ]:
%%bash
# 10b · Encode a brain the model has NEVER seen (zero-shot new subject).
# The encoder takes no subject-ID; identity enters only at the normaliser, so a new person needs
# only a short UNLABELLED baseline to compute their own scale/covariance — no labels, no retraining.
# Demonstrated on the held-out subject ZAB. Runs in the uv env (which has zte).
uv run python - <<'PY'
import os
from zte.config import ZTEConfig
from zte.data.dataset import ZuCoDataset
from zte.inference.embed import ZTEEmbedder

DATA_DIR, DRIVE_DIR = os.environ['DATA_DIR'], os.environ['DRIVE_DIR']
HOLDOUT = 'ZAB'
run_dir = f'{DRIVE_DIR}/experiments/exp8_clip_e5_lo{HOLDOUT}'
emb = ZTEEmbedder.from_checkpoint(f'{run_dir}/checkpoints/best.pt')

# Rebuild the exact feature pipeline the run used, UN-normalised (the embedder normalises):
dcfg = ZTEConfig.from_yaml(f'{run_dir}/config.yaml').dataset
dcfg.root, dcfg.normalize = DATA_DIR, 'none'
ds = ZuCoDataset(dcfg).build(show_progress=False)
mask = (ds.words['subject'].to_numpy() == HOLDOUT) & ds.presence
feats = ds.features[mask]                        # (n, in_dim) raw features at the model's width
baseline_bp, words_bp = feats[:200], feats[200:]   # first ~200 words = unlabelled baseline
emb.calibrate_subject(baseline_bp, subject_code=HOLDOUT)
vectors = emb.embed_signals(band_power=words_bp, subject_codes=[HOLDOUT] * len(words_bp), show_progress=False)
print('encoded', vectors.shape, 'thought vectors for a calibrated new brain')
PY

## 11 · Housekeeping (free space / fresh clone)
Colab disk is small. Remove individual runs with **`remove_from_res('run_name')`** (defined in Section 4), or delete whole `res/` subtrees with `zte-pack clean`; or wipe the checkout and re-clone. In every case your **data and saved runs on Drive are untouched** — only local scratch is removed.

In [ ]:
# Remove specific runs locally (easy; does NOT touch Drive). Frees space after a smoke test:
remove_from_res('colab_exp6', 'colab_exp6_spatial')  # bare run names under res/experiments/
# remove_from_res('res/benchmark', 'res/cache')        # or any res/ subpath

# Or free whole res/ subtrees via the CLI (dry-run without --yes):
# !uv run zte-pack clean experiments cache benchmark --yes
# Wipe everything under res/:  !uv run zte-pack clean all --yes

In [ ]:
# Fresh clone — wipe the checkout and re-clone (e.g. after pushing critical updates).
# Your data + saved runs on Google Drive are NOT touched.
%cd /content
!rm -rf zte
!git clone --depth 1 https://github.com/victor-iyi/zte.git --branch main
%cd zte
!uv sync --group all
# Then re-run Section 2 (bootstrap) and re-mount Drive (Section 4).